# Level 1 — Lesson 4: Debugging and Reviewing LLM-Generated Code

## Does the code only run, or is it actually good?

**Teaching time:** ~40–45 minutes  
**Core question:** *How do I debug and review LLM-generated code without simply asking the LLM to rewrite everything?*

In Lesson 3 we built a small segmentation pipeline with help from an LLM.

Now assume something important:

> **The code runs.**

That is useful, but not sufficient.

Generated code can still be:

```text
unnecessarily complicated
logically wrong
scientifically inappropriate
difficult to reproduce
hiding errors instead of solving them
```

The goal of this lesson is to learn how to review LLM code, not only how to make errors disappear.

# 0. Four kinds of problems

It helps to separate different failure modes.

### 1. Execution error

The program crashes.

```text
NameError
TypeError
CUDA out of memory
shape mismatch
missing package
```

### 2. Logical error

The program runs, but it does not do what you intended.

```text
wrong axis
wrong file
wrong condition
wrong output
```

### 3. Scientific error

The implementation is computationally valid, but the method is inappropriate.

```text
continuous interpolation for a segmentation label
patient leakage between training and validation
an augmentation that changes clinical meaning
test data used to tune the pipeline
```

### 4. Unnecessary complexity

The program may even be correct, but harder to understand than necessary.

```text
wrapper functions around one library call
classes with no useful state
unused configuration
duplicated conversions
comments that only repeat Python
```

> **The last three are often more dangerous than a syntax error because Python may never warn you.**

# 1. Be precise and concise in you fix request

A poor debugging workflow is:

```text
error
→ paste everything into LLM
→ "fix it"
→ replace large parts of the program
→ hope
```

A better workflow is:

```text
reproduce
↓
read the error
↓
find the failing line
↓
inspect shapes / types / values
↓
form a hypothesis
↓
make the smallest change
↓
rerun the relevant test
↓
run cleanly from the beginning
```

A useful prompt is:

> This code fails on the line shown below.  
> **Expected behaviour:** ...  
> **Actual behaviour:** ...  
> **Relevant shapes/types:** ...  
> **Error:** ...  
>
> First explain the likely cause. Then suggest the smallest change that would test your hypothesis. Do not rewrite unrelated code.

The computer should tell you whether the proposed fix worked.

# 2. Do not simply delete the error or its check

An easy 'fix' would be to remove the line of the code that is erroneous.  For example, as a user with no experience, you may get the problematic code to run, by deleting an assertion or a try-catch block.

Assertions and checks are often there because an assumption matters.

Let's deliberately create a mismatch.

In [1]:
import numpy as np
import torch
import torch.nn.functional as F

prediction = torch.zeros((1, 1, 64, 64, 32))
reference  = torch.zeros((1, 1, 64, 63, 32))

print("Prediction:", tuple(prediction.shape))
print("Reference: ", tuple(reference.shape))

Prediction: (1, 1, 64, 64, 32)
Reference:  (1, 1, 64, 63, 32)


In [2]:
assert prediction.shape == reference.shape, (
    f"Prediction and reference must match, "
    f"but got {tuple(prediction.shape)} and {tuple(reference.shape)}"
)

AssertionError: Prediction and reference must match, but got (1, 1, 64, 64, 32) and (1, 1, 64, 63, 32)

The assertion fails.

Imagine you or your LLM get this idea:

```text
The assertion is causing the error, so remove it.
```

Does this solve the problem?

No, of course not. It removes the code that **detected** it.

A better response is:

> The assertion correctly shows that prediction and reference have different spatial shapes. Do not remove the assertion. Help me identify where the one-voxel difference could have been introduced and what I should inspect upstream.

The same principle applies to code such as:

```python
try:
    ...
except Exception:
    pass
```

A 'fix' that hides a problem may be worse than the original failure.

> **Do not only check for the line that caused an error, but also the part of the code before.**

In aboves code, the exception looks like this:

```python
---------------------------------------------------------------------------

AssertionError   Traceback (most recent call last)

/tmp/ipykernel_504/1656893536.py in <cell line: 0>()
----> 1 assert prediction.shape == reference.shape, (
      2     f"Prediction and reference must match, "
      3     f"but got {tuple(prediction.shape)} and {tuple(reference.shape)}"
      4 )

AssertionError: Prediction and reference must match, but got (1, 1, 64, 64, 32) and (1, 1, 64, 63, 32)
```

The line that *raises* the error is  given as:

```python
<cell line: 0>
```

But the error was caused earlier when setting up the arrays.

### Do you expect such error 'fixing' from LLMs?

Not really. It is highly *unlikely* that you will ever see somethig so blatantly obviously wrong from a contemporary LLM. Even the lower quality models perform reasonably well.

> However, this type of *issue* is common!

Consider the following fixes to the code above:

```python
prediction = torch.zeros((1, 1, 64, 63, 32))
reference  = torch.zeros((1, 1, 64, 63, 32))
```

or

```python
prediction = torch.zeros((1, 1, 64, 64, 32))
reference  = torch.zeros((1, 1, 64, 64, 32))
```

Which one is correct? Is either correct?

Both fix the error, but may not be correct. Maybe there is also specific *reason* for the shape mismatch. If you are not specific, the LLM will make that assumption for you based on the *most likely* case.  

> **Always check the underlying assumptions together with the problem**


# 3. Scientific debugging: executable code can still be wrong

Segmentation masks contain **categories**, not continuous measurements.

Create a small binary mask:

In [3]:
label = torch.zeros((1, 1, 8, 8))
label[:, :, 2:6, 2:6] = 1

print("Original values:", torch.unique(label).tolist())

Original values: [0.0, 1.0]


In [4]:
label_bilinear = F.interpolate(
    label,
    size=(13, 13),
    mode="bilinear",
    align_corners=False,
)

label_nearest = F.interpolate(
    label,
    size=(13, 13),
    mode="nearest",
)

print("Bilinear values:")
print(torch.unique(label_bilinear))

print("\nNearest-neighbour values:")
print(torch.unique(label_nearest))

Bilinear values:
tensor([0.0000, 0.0015, 0.0015, 0.0015, 0.0251, 0.0251, 0.0251, 0.0251, 0.0385,
        0.0385, 0.4275, 0.4275, 0.4275, 0.6538, 0.6538, 1.0000])

Nearest-neighbour values:
tensor([0., 1.])


###Exercise: Which of the two implementations is correct for a segmentation task?

Pause and reflect on the code. You can ask your LLM to explain it to you, but specifically ask it also to not comment on the validity of either approach yet!

> **Which is the right approach for a segmentation task**?

Both operations are valid PyTorch.

But our labels mean:

```text
0 = background
1 = spleen
```

What does a value such as `0.0385` mean as a class label?

Nothing useful.

The bilinear operation did exactly what was requested. However, the **request was inappropriate for the data**.

Provide the code to your LLM and ask:

> Both implementations execute successfully. Explain why one interpolation method is inappropriate for a binary segmentation label. Do not judge them only by whether the code runs.

> **Technical validity and scientific validity are different things.**

# 4. Another silent error: patient leakage

Imagine several image patches were extracted from each patient.

The following example is entirely synthetic.

In [ ]:
patches = []

for patient_id in range(6):
    for patch_id in range(5):
        patches.append({
            "patient": f"patient_{patient_id}",
            "patch": f"patch_{patch_id}",
        })

rng = np.random.default_rng(42)
rng.shuffle(patches)

split = int(0.8 * len(patches))
train_patches = patches[:split]
val_patches = patches[split:]

train_patients = {x["patient"] for x in train_patches}
val_patients = {x["patient"] for x in val_patches}

print("Training patients:  ", sorted(train_patients))
print("Validation patients:", sorted(val_patients))
print("Overlap:            ", sorted(train_patients & val_patients))

The split is random and the proportions look reasonable.

But the same patient appears in training and validation.

That is **data leakage**.

The generic pattern:

```python
shuffle(data)
train = data[:80]
val = data[80:]
```

is fine for many datasets.

It becomes wrong when more than one sample comes from the same patient.

The safer order is:

```text
patients
↓ split
training patients / validation patients
↓
extract or select patches
```

> **A common coding pattern can be wrong for an uncommon scientific task.**

In [ ]:
patient_ids = [f"patient_{i}" for i in range(6)]

rng = np.random.default_rng(42)
rng.shuffle(patient_ids)

train_patient_ids = set(patient_ids[:5])
val_patient_ids = set(patient_ids[5:])

print("Training patients:  ", sorted(train_patient_ids))
print("Validation patients:", sorted(val_patient_ids))
print("Overlap:            ", sorted(train_patient_ids & val_patient_ids))

# 5. Fluff: abstraction without a reason

Consider this code:

In [ ]:
class ImageRotationConfiguration:
    def __init__(self, number_of_rotations=1):
        self.number_of_rotations = number_of_rotations


class ImageRotationProcessor:
    def __init__(self, configuration):
        self.configuration = configuration

    def validate_input(self, image):
        if not isinstance(image, np.ndarray):
            raise TypeError("Input must be a numpy array.")

    def perform_rotation(self, image):
        self.validate_input(image)
        return np.rot90(
            image,
            k=self.configuration.number_of_rotations,
        )


def create_rotation_configuration(number_of_rotations=1):
    return ImageRotationConfiguration(number_of_rotations)


def rotate_image_using_processor(image, number_of_rotations=1):
    config = create_rotation_configuration(number_of_rotations)
    processor = ImageRotationProcessor(config)
    return processor.perform_rotation(image)

In [ ]:
image = np.arange(9).reshape(3, 3)

result = rotate_image_using_processor(image, 1)
print(result)

There is nothing inherently wrong with:

- classes;
- validation;
- configuration;
- helper functions.

The question is whether they solve a real problem.

The required operation is:

```python
np.rot90(image, k=1)
```

The generated version adds several concepts without meaningful extra capability.

This is what we mean by **fluff**.

Ask your LLM:

> Simplify this code with a strong bias toward deletion.  
> The only required behaviour is to rotate a NumPy array by `k * 90°`.  
>
> Before rewriting:
> 1. identify what can be removed;
> 2. explain whether removal changes behaviour;
> 3. propose the smallest implementation;
> 4. preserve the same input/output behaviour.

In [ ]:
def rotate_image(image, k=1):
    return np.rot90(image, k=k)

Even the function above is still unnecessary in most contexts, and could simply be a library call:

```python
rotated = np.rot90(image, k=1)
```

> **Abstraction is useful when it hides meaningful complexity or represents a reusable concept.**

LLM geneated code often has these abstractions or duplicate things that are already there.


# 6. 'Clean up this code' is too vague

If you ask:

> Clan up this code.

what does *clean* mean?

```text
shorter?
more modular?
more object-oriented?
more defensive?
more configurable?
more reusable?
less comments?
more comments?
```

A more useful prompt is:

> Make this code easier for a beginner to audit. Preserve the exact behaviour. Prefer direct NumPy/PyTorch/MONAI calls over wrappers. Remove functions used only once unless their name adds useful meaning. Do not add new functionality.

> **Explain the LLM what you want.**

# 7. Comments can also be fluff

Comments are for other people to be able to follow and understand the logic in your code. They should primarily help to make the code more human readable. Comments are also good to highlight 'TODOs', or explain certain choices and potential failures. LLMs like to comment **a lot**. But while some comments are always better than none, adding more or extensive comments is not always improving readabiltiy.    

Compare:

```python
# Add one to x
x += 1
```

with:

```python
# Labels are categorical, so use nearest-neighbour interpolation.
```

The first comment only translates Python into English. The second explains why a scientific choice was made. And even that comment is not strictly necessary for someone else to understand a piece of code, where the interpolations are executed.

Ask whether a comments does explain:

```text
why
a non-obvious assumption
a scientific choice
a limitation
a workaround
was made
```

A useful review instruction is:

> Remove comments that merely translate Python into English. Keep comments that explain scientific assumptions, non-obvious choices, or limitations.

In addition, code comments should strictly be in English. Some LLMs, however, comment in the set user language, which might be English, but might also be your native language.

###Common pitfal for large projects
LLMs sometimes add preambles to files in larger coding projects. This is in principle a good practice. However, it can also be the biggest tell that code is purely LLM generated an was not looked at by the user.

Why?

Because I often find something like
```python
# author: YOUR NAME HERE
# team: YOUR TEAMS NAME HERE
```

Verify that you have changed comments to be meaningful.

Also, it is important to check that the LLM did not disclose information you did not want it to disclose in your code before you make it publicly available.

Examples can include:
```text
user names
local file directories
team member names
email addresses
```

This can lead to critical leakage of private data.

> **Never push code to a public repository that you have not checked for  critical information!**


# 8. Defensive programming can invent answers

LLMs frequently try to make code 'robust' and catch invalid user input. This is good practice! However, it may check for cases that can never happen, or more problematically, it may introduce a default behavior that is totally unwanted.

Consider:

In [ ]:
def unsafe_mean(values):
    try:
        return sum(values) / len(values)
    except Exception:
        return 0

print("Mean of []:", unsafe_mean([]))

The function never crashes.

But is the mean of an empty dataset really `0`?

No.

The fallback creates a plausible number with the wrong meaning.

For scientific code, a visible failure may be safer:

```python
if len(values) == 0:
    raise ValueError("Cannot compute mean of an empty sequence.")
```

> **Robust software should handle expected situations. It should not silently invent scientifically meaningful-looking values.**

# 9. Large rewrites create new uncertainty

Suppose one line fails in a 100-line program.

You ask:

> Fix the error.

The LLM returns 170 lines and also changes:

```text
data loading
model architecture
optimizer
plotting
output format
```

Even if the error disappears, you no longer know which change fixed it.

A better prompt is:

> Identify the likely cause first. Then provide the smallest possible patch. Do not modify unrelated code. Show exactly which lines need to change and why.

Small patches are easier to understand, test, and undo.

# 10. Create tests to preserve what must remain stable

Sometimes, a rewrite of part of the code silently changes the behavior or even the scientific accuracy of the total code. It is therefore (and in general in coding projects) good practice to establish 'tests' that show the code is still working as intended. This can be done entirely by the LLM or you provide example tests to the LLM that it should consider in generating the code.

In [ ]:
test_image = np.arange(12).reshape(3, 4)

old_result = rotate_image_using_processor(
    test_image,
    number_of_rotations=2,
)

new_result = rotate_image(
    test_image,
    k=2,
)

assert np.array_equal(old_result, new_result)

print("Behaviour preserved.")

Tests are especially useful when an LLM refactors code, because a visually elegant rewrite can silently change behaviour.

Useful scientific checks include:

```text
expected shapes
expected label values
known numerical examples
image/label alignment
patient-split independence
expected physical units
known edge cases
```


#11. Use the LLM as a reviewer

In [ ]:
import os
import torch
import monai
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Spacingd,
    Resized,
    ScaleIntensityRanged,
)
# ==========================================
# Configuration class
# ==========================================
class SpleenDataPipelineConfiguration:
    """Class to hold configuration details for Spleen segmentation pipeline."""
    def __init__(self,
                 image_key="image",
                 label_key="label",
                 target_spatial_shape=(128, 128, 128)):
        # Set the image key name variable
        self.image_key = image_key
        # Set the label key name variable
        self.label_key = label_key
        # Set the target spatial shape tuple
        self.target_spatial_shape = target_spatial_shape

    def get_image_key(self):
        # Returns the image key
        return self.image_key

    def get_label_key(self):
        # Returns the label key
        return self.label_key

    def get_target_spatial_shape(self):
        # Returns the spatial shape tuple
        return self.target_spatial_shape

# Instantiating the configuration object
config = SpleenDataPipelineConfiguration()

# ==========================================
# Create the MONAI transform compose
# Load images, ensure channel first,
# Scale the CT intensity
# Resize CT and label
# ==========================================
transforms = Compose([
    # Load the dictionary-based medical image
    LoadImaged(keys=[config.get_image_key(), config.get_label_key()]),
    # Ensure there is a channel dimension first in the tensor representation
    EnsureChannelFirstd(keys=[config.get_image_key(), config.get_label_key()]),
    # Normalize intensity values specifically for spleen CT values
    ScaleIntensityRanged(
        keys=[config.get_image_key()],
        a_min=-164,
        a_max=-57,
        b_min=0.0,
        b_max=1.0,
        clip=True,
    ),
    # Resizing transforms:
    Resized(
        keys=[config.get_image_key()],
        spatial_size=config.get_target_spatial_shape(),
        mode="bilinear",
    ),
    Resized(
        keys=[config.get_label_key()],
        spatial_size=(128, 128, 64),
        mode="bilinear",
    ),
])

# Dummy input data mimicking Medical Decathlon format
dummy_dataset = [{
    "image": torch.randn(1, 256, 256, 100).numpy(),
    "label": (torch.randn(1, 256, 256, 100) > 0.5).float().numpy(),
}]

print("Processing dummy dataset sample...")
try:
    # Transform the dummy sample through the dictionary pipeline
    processed_sample = transforms(dummy_dataset[0])

    img_shape = processed_sample[config.get_image_key()].shape
    lbl_shape = processed_sample[config.get_label_key()].shape

    print(f"Processed Image Shape: {img_shape}")
    print(f"Processed Label Shape: {lbl_shape}")

    # Assert match
    assert img_shape == lbl_shape, f"Mismatch! Image: {img_shape}, Label: {lbl_shape}"

except Exception as e:
    print(f"Pipeline execution failed with error: {e}")



Try this on the code above:

> Review this code as a scientific Python reviewer. Do not rewrite it yet.  
>
> Classify findings into:
>
> **A. Correctness problems**  
> Code that likely does not do what was intended.
>
> **B. Scientific assumptions**  
> Choices that may be technically valid but require justification.
>
> **C. Unnecessary complexity**  
> Code that could be removed or simplified without changing behaviour.
>
> **D. Reproducibility problems**  
> Hidden state, randomness, environment/version assumptions, or steps that fail from a clean runtime.
>
> For each finding:
> - identify the relevant operation;
> - explain why it matters;
> - suggest the smallest change;
> - state what evidence would verify the change.
>
> Do not rewrite unrelated sections.



## 12. Do not assume agreement means validation

Questions such as:

```text
"Am I right that...?"
"Wouldn't it be better to...?"
"Do you agree that...?"
```

invite agreement.

Instead of:

> Isn't bilinear interpolation better because it is smoother?

ask:

> Compare bilinear and nearest-neighbour interpolation for a binary segmentation mask. State the consequences of each and identify which preserves categorical labels.

Or:

> Try to find reasons why my proposed approach could be wrong.

> **Agreement from an LLM is not independent evidence.**

# 13. Final exercise: review before asking an LLM

Consider:

```python
patches = extract_patches(all_patients)
random.shuffle(patches)

train = patches[:800]
validation = patches[800:]

def resize_mask(mask):
    return torch.nn.functional.interpolate(
        mask.float(),
        size=(64, 64, 64),
        mode="trilinear",
        align_corners=False,
    )

try:
    validation_dice = calculate_dice(model, validation)
except Exception:
    validation_dice = 0.0
```

### First review it yourself

Try to identify at least three concerns.

Think about:

```text
patient independence
categorical labels
error handling
```

### Then ask the LLM

> Review this code for scientific and computational problems. Do not rewrite it. For each problem, explain what assumption is violated and what evidence would show that the problem has been fixed.

Compare its review with your own.

Did it find everything?

Did it invent a problem that is not actually present?

# 14. Compact review checklist

Before accepting generated scientific code:

### Does it run?
No syntax/runtime errors? Reasonable memory use? Dependencies available?

### Does it do what I asked?
Correct inputs, outputs, shapes, units, files?

### Does the method make scientific sense?
Appropriate preprocessing, interpolation, split, metric, assumptions?

### Is the complexity justified?
Does each function/class/configuration add something useful?

### Did the fix change anything unrelated?
Data, model, loss, evaluation, randomness, output?

### Does it reproduce?
Restart the runtime and run everything from the beginning.

# Take-home message

Treat LLM-generated code like code from a very fast collaborator:

```text
useful
productive
often knowledgeable
but still requires review
```

A good debugging workflow is:

```text
reproduce
↓
inspect
↓
understand
↓
change as little as possible
↓
test
↓
simplify
↓
restart and reproduce
```

Errors in LLM generated code take all shapes and forms. Sometimes the LLM may invent a function that *would fit* the remaining program, but does not actually exist. Sometimes it may do that with library functions. Sometimes, that API call the LLM codes is for an older version of the library. All this is real.

> **AI can not only generate code faster than any human, it can also create more problems than any human reviewer can easily catch**

Have the LLM plan. Build one chunk of the project at a time. Review and verify. Do not assume just because the LLM *can* build a huge codebase in seconds, that its content is really what you wanted.